In [ ]:
# CELL 1: Install packages
# This cell installs only the packages needed for this beginner notebook.
# Run this once before running the remaining cells.

%pip install -q openai==2.44.0 langchain-openai==1.3.0 langgraph==1.2.9 langsmith==0.10.10

In [ ]:
# CELL 2: Configure credentials and imports
# Keys are entered at runtime so secrets are not stored in the notebook.
# Use environment variables, notebook secrets, Azure Key Vault, or another secret store for shared projects.
# LangSmith tracing is enabled so every graph node and LLM call can be observed in LangSmith.

import json
import os
import re
import sys
from getpass import getpass
from typing import Literal, TypedDict
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langsmith import traceable


# Windows terminals sometimes cannot print special characters returned by the model.
# This keeps notebook/terminal output readable.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")


# Helper function: read from environment first, then ask the learner.
# Hidden input is used for API keys so they are not displayed in notebook output.
def read_secret(setting_name: str, prompt_text: str, hidden: bool = False, default: str = "") -> str:
    value = os.environ.get(setting_name, "").strip()
    if value:
        return value

    display_prompt = f"{prompt_text}"
    if default:
        display_prompt = f"{prompt_text} [{default}]: "
    value = getpass(display_prompt).strip() if hidden else input(display_prompt).strip()
    if not value and default:
        value = default
    if not value:
        raise ValueError(f"{setting_name} is required to run this notebook.")
    os.environ[setting_name] = value
    return value


# Azure OpenAI Foundry model settings.
os.environ["AZURE_OPENAI_ENDPOINT"] = read_secret(
    "AZURE_OPENAI_ENDPOINT",
    "Azure OpenAI endpoint, for example https://your-resource.openai.azure.com/openai/v1: ",
)
os.environ["AZURE_OPENAI_API_KEY"] = read_secret(
    "AZURE_OPENAI_API_KEY",
    "Azure OpenAI API key, hidden input: ",
    hidden=True,
)
os.environ["AZURE_OPENAI_API_VERSION"] = read_secret(
    "AZURE_OPENAI_API_VERSION",
    "Azure OpenAI API version",
    default="2025-08-07",
)
os.environ["AZURE_OPENAI_DEPLOYMENT"] = read_secret(
    "AZURE_OPENAI_DEPLOYMENT",
    "Azure OpenAI deployment name",
    default="gpt-5-mini",
)

# External tool keys.
os.environ["OPENWEATHER_API_KEY"] = read_secret(
    "OPENWEATHER_API_KEY",
    "OpenWeatherMap API key, hidden input: ",
    hidden=True,
)
os.environ["SERPER_API_KEY"] = read_secret(
    "SERPER_API_KEY",
    "Serper API key, hidden input: ",
    hidden=True,
)

# LangSmith tracing settings.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = read_secret(
    "LANGSMITH_ENDPOINT",
    "LangSmith endpoint",
    default="https://api.smith.langchain.com",
)
os.environ["LANGSMITH_API_KEY"] = read_secret(
    "LANGSMITH_API_KEY",
    "LangSmith API key, hidden input: ",
    hidden=True,
)
os.environ["LANGSMITH_PROJECT"] = read_secret(
    "LANGSMITH_PROJECT",
    "LangSmith project name",
    default="lab20_langgraph_tool_agent",
)


print("Credentials configured. LangSmith project:", os.environ["LANGSMITH_PROJECT"])


In [ ]:
# CELL 3: Create the model, state, and tools
# The model is used by the router agent and by the final response generator.
# The tools are plain Python functions. LangGraph decides which one should run.

llm = ChatOpenAI(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)


class AgentState(TypedDict):
    # User question entered in the notebook.
    question: str

    # Route selected by the router agent: calculator, weather, search, or general.
    route: str

    # Name of the tool that was used.
    tool_name: str

    # Raw result returned by the selected tool.
    tool_result: str

    # Final user-friendly answer.
    final_answer: str


@traceable(name="calculator_tool", run_type="tool")
def calculator(expression: str) -> str:
    """Calculate a basic math expression safely."""
    allowed_pattern = r"^[0-9+\-*/().\s]+$"
    if not re.match(allowed_pattern, expression):
        return "Invalid expression. Use only numbers and + - * / ( )."

    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Calculation error: {exc}"


@traceable(name="weather_tool", run_type="tool")
def get_weather(city: str) -> str:
    """Call OpenWeatherMap and return current weather for one city."""
    params = urlencode(
        {
            "q": city,
            "appid": os.environ["OPENWEATHER_API_KEY"],
            "units": "metric",
        }
    )
    url = f"https://api.openweathermap.org/data/2.5/weather?{params}"

    try:
        with urlopen(url, timeout=15) as response:
            data = json.loads(response.read().decode("utf-8"))
        return (
            f"{data['name']}, {data['sys']['country']}: "
            f"{data['weather'][0]['description']}, "
            f"{data['main']['temp']} C, humidity {data['main']['humidity']}%."
        )
    except Exception as exc:
        return f"Weather API error: {exc}"


@traceable(name="serper_search_tool", run_type="tool")
def web_search(query: str) -> str:
    """Call Serper API and return the top web search results."""
    request = Request(
        "https://google.serper.dev/search",
        data=json.dumps({"q": query, "num": 3}).encode("utf-8"),
        headers={
            "X-API-KEY": os.environ["SERPER_API_KEY"],
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urlopen(request, timeout=15) as response:
            data = json.loads(response.read().decode("utf-8"))
    except Exception as exc:
        return f"Serper API error: {exc}"

    results = data.get("organic", [])[:3]
    if not results:
        return "No search results found."

    lines = []
    for index, item in enumerate(results, start=1):
        lines.append(
            f"{index}. {item.get('title', 'No title')}\n"
            f"{item.get('snippet', 'No snippet')}\n"
            f"{item.get('link', 'No link')}"
        )
    return "\n\n".join(lines)


print("Model and tools are ready.")

In [ ]:
# CELL 4: Create the router agent
# The router agent is an LLM call that chooses which path the graph should follow.
# It returns one of four route names: calculator, weather, search, or general.

RouteName = Literal["calculator", "weather", "search", "general"]


@traceable(name="router_agent", run_type="chain")
def router_agent(question: str) -> RouteName:
    system_prompt = (
        "You are a routing agent. Choose the best route for the user question. "
        "Return only one word: calculator, weather, search, or general. "
        "Use calculator for math. Use weather for city weather. "
        "Use search for current/latest/web information. Use general otherwise."
    )
    response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ]
    )
    route = (response.content or "general").strip().lower()
    if route not in {"calculator", "weather", "search", "general"}:
        route = "general"
    return route  # type: ignore[return-value]


def router_node(state: AgentState) -> AgentState:
    state["route"] = router_agent(state["question"])
    return state


def choose_route(state: AgentState) -> str:
    return state["route"]


print("Router agent is ready.")

In [ ]:
# CELL 5: Create graph nodes for each tool and final answer generation
# Each branch updates the shared state. LangGraph passes this state between nodes.

def extract_math_expression(question: str) -> str:
    # Keep only math-safe characters from the question.
    expression = "".join(ch for ch in question if ch in "0123456789+-*/(). ")
    return expression.strip() or "0"


def extract_city(question: str) -> str:
    # Beginner-friendly extraction: use words after 'in', otherwise default to Delhi.
    lowered = question.lower()
    if " in " in lowered:
        return question.split(" in ")[-1].strip(" ?.!")
    return "Delhi"


def calculator_node(state: AgentState) -> AgentState:
    expression = extract_math_expression(state["question"])
    state["tool_name"] = "calculator"
    state["tool_result"] = calculator(expression)
    return state


def weather_node(state: AgentState) -> AgentState:
    city = extract_city(state["question"])
    state["tool_name"] = "weather"
    state["tool_result"] = get_weather(city)
    return state


def search_node(state: AgentState) -> AgentState:
    state["tool_name"] = "serper_search"
    state["tool_result"] = web_search(state["question"])
    return state


def general_node(state: AgentState) -> AgentState:
    state["tool_name"] = "no_tool"
    state["tool_result"] = "No external tool was needed."
    response = llm.invoke(
        [
            {"role": "system", "content": "Answer clearly in 3-5 sentences."},
            {"role": "user", "content": state["question"]},
        ]
    )
    state["final_answer"] = response.content or ""
    return state


@traceable(name="final_answer_agent", run_type="chain")
def answer_node(state: AgentState) -> AgentState:
    # This node converts raw tool output into a friendly final answer.
    response = llm.invoke(
        [
            {"role": "system", "content": "Use the tool result to answer the user clearly and briefly."},
            {
                "role": "user",
                "content": (
                    f"Question: {state['question']}\n"
                    f"Tool used: {state['tool_name']}\n"
                    f"Tool result:\n{state['tool_result']}"
                ),
            },
        ]
    )
    state["final_answer"] = response.content or ""
    return state


print("Tool nodes and answer node are ready.")

In [ ]:
# CELL 6: Build the LangGraph workflow
# This graph starts at the router. Based on the route, it branches to one tool.
# Tool branches then go to the final answer node. General questions skip tools.

def build_graph():
    graph = StateGraph(AgentState)

    graph.add_node("router", router_node)
    graph.add_node("calculator", calculator_node)
    graph.add_node("weather", weather_node)
    graph.add_node("search", search_node)
    graph.add_node("general", general_node)
    graph.add_node("answer", answer_node)

    graph.set_entry_point("router")
    graph.add_conditional_edges(
        "router",
        choose_route,
        {
            "calculator": "calculator",
            "weather": "weather",
            "search": "search",
            "general": "general",
        },
    )

    graph.add_edge("calculator", "answer")
    graph.add_edge("weather", "answer")
    graph.add_edge("search", "answer")
    graph.add_edge("answer", END)
    graph.add_edge("general", END)

    return graph.compile()


workflow = build_graph()

# Mermaid diagram can be pasted into https://mermaid.live if learners want a visual graph.
print(workflow.get_graph().draw_mermaid())

In [ ]:
# CELL 7: Run the routing agent
# Change the question variable to test different branches.
# After running, open LangSmith and check project: lab20_beginner_langgraph_tool_agent.

@traceable(name="beginner_langgraph_tool_routing_agent", run_type="chain")
def run_question(question: str) -> AgentState:
    return workflow.invoke(
        {
            "question": question,
            "route": "",
            "tool_name": "",
            "tool_result": "",
            "final_answer": "",
        }
    )


question = "What is the weather in Delhi?"
result = run_question(question)

print("Question:", result["question"])
print("Route selected:", result["route"])
print("Tool used:", result["tool_name"])
print("Tool result:\n", result["tool_result"])
print("Final answer:\n", result["final_answer"])